In [1]:
##### Compiles rasters of all predictors into one file

import os
import pandas as pd
import rasterio
from pathlib import Path
import numpy as np
from scipy.spatial import cKDTree

In [2]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent 

# folder of predictors 
predictors = Path(f'{cd}/Data/Clean/Predictors/Rasters')
tif_files  = sorted(predictors.glob("*.tif"))

# USD production 
production_path = f'{cd}/Data/Clean/Production/total_production_USD_2020.tif'

# save path
save_path_predictors = f'{cd}/Data/Clean/Training_data/raster_predictor_matrix.parquet'
save_path_production = f'{cd}/Data/Clean/Training_data/raster_production_matrix.parquet'

In [3]:
#### Combine all rasters

# read rasters
arrays = {}
coords = None

for path in tif_files:
    with rasterio.open(path) as src:
        arr    = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        if coords is None:
            height, width = arr.shape
            transform = src.transform
            rows, cols = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
            xs, ys = rasterio.transform.xy(transform, rows.ravel(), cols.ravel())
            coords = pd.DataFrame({"x": xs, "y": ys})

        arrays[path.stem] = arr.ravel()

# add production raster
with rasterio.open(production_path) as src:
    arr    = src.read(1).astype(np.float32)
    nodata = src.nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan
    arrays["total_production_USD_2020"] = arr.ravel()

# compile into dataframe
all_rasters = pd.concat([coords, pd.DataFrame(arrays)], axis=1)

# drop if terrain is missing (selects only land pixels)
all_rasters = all_rasters.dropna(subset=["terrain_slope"])
all_rasters = all_rasters.dropna(subset=["total_production_USD_2020"])
all_rasters = all_rasters[all_rasters['total_production_USD_2020'] != 0]

In [4]:
### Fill in missing data 

# define function which fill's NaN's with value from nearest pixel that has data (by Euclidean distance in x/y space)
def fill_missing_nearest(df, coord_cols=("x", "y")):
    
    data_cols = [c for c in df.columns if c not in coord_cols]
    coords_arr = df[list(coord_cols)].values

    for col in data_cols:
        n_missing = df[col].isna().sum()
        if n_missing == 0:
            continue

        print(f"  Filling {n_missing:,} missing values in '{col}'...")

        has_data = ~df[col].isna()

        # build tree from pixels that have data for this column
        tree = cKDTree(coords_arr[has_data])

        # query tree for each missing pixel
        missing_mask = ~has_data
        _, idx = tree.query(coords_arr[missing_mask], workers=-1)

        # map back to donor values
        donor_values = df.loc[has_data, col].values[idx]
        df.loc[missing_mask, col] = donor_values

    return df

all_rasters = fill_missing_nearest(all_rasters)

  Filling 39 missing values in 'GDP_per_capita'...
  Filling 6 missing values in 'child_dependency_ratio'...
  Filling 20,388 missing values in 'country_capital_intensity'...
  Filling 31 missing values in 'country_labor_intensity'...
  Filling 6 missing values in 'crop_intensity'...
  Filling 258 missing values in 'economic_objective_prob'...
  Filling 6 missing values in 'female_share'...
  Filling 4,323 missing values in 'field_size_share_large'...
  Filling 4,323 missing values in 'field_size_share_medium'...
  Filling 4,323 missing values in 'field_size_share_small'...
  Filling 4,323 missing values in 'field_size_share_vlarge'...
  Filling 4,323 missing values in 'field_size_share_vsmall'...
  Filling 20,388 missing values in 'log_country_capital_intensity'...
  Filling 31 missing values in 'log_country_labor_intensity'...
  Filling 4 missing values in 'pct_GDP_ag'...
  Filling 66 missing values in 'pct_cropland_irrigated'...
  Filling 3 missing values in 'season_length'...
  Fil

In [5]:
#### Drop compositional variables

# one of each of field size %'s and production mix %'s 
# share_vlarge_field, other_share_production_USD  

# also split predictors and target

col_to_drop = ['total_production_USD_2020', 'field_size_share_vlarge', 'other_share_production_USD']
predictors = all_rasters.drop(columns = col_to_drop)

production = all_rasters[['x', 'y', 'total_production_USD_2020']]

In [6]:
# save
predictors.to_parquet(save_path_predictors, index=False)
production.to_parquet(save_path_production, index=False)